In [ ]:
# In[1]:

import pandas as pd
import pytz

# timezone for UTC+8
tz = pytz.timezone('Asia/Shanghai')

# Load CSVs (they are in current working directory)
df_metric = pd.read_csv('metric.csv')
df_trace = pd.read_csv('trace.csv')
df_log = pd.read_csv('log.csv')
df_error = pd.read_csv('error_logs.csv')

# Helper to add localized timestamp column 'ts' (UTC+8) while keeping original epoch 'timestamp'
def add_local_ts(df):
    # convert epoch seconds to UTC then to Asia/Shanghai
    df['ts'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    return df

df_metric = add_local_ts(df_metric)
df_trace = add_local_ts(df_trace)
df_log = add_local_ts(df_log)
df_error = add_local_ts(df_error)

# Generic summarizer
def summarize_file(df, id_col, name_col=None, sample_cols=None, distinct_limit=50, sample_limit=20):
    row_count = int(df.shape[0])
    distinct_ids = pd.Series(sorted(df[id_col].dropna().unique())).head(distinct_limit).reset_index(drop=True).to_frame(name=id_col)
    distinct_ids_count = int(df[id_col].nunique(dropna=True))
    if name_col:
        distinct_names = pd.Series(sorted(df[name_col].dropna().unique())).head(distinct_limit).reset_index(drop=True).to_frame(name=name_col)
        distinct_names_count = int(df[name_col].nunique(dropna=True))
    else:
        distinct_names = None
        distinct_names_count = None
    if row_count > 0:
        global_min_ts = df['ts'].min()
        global_max_ts = df['ts'].max()
    else:
        global_min_ts = pd.NaT
        global_max_ts = pd.NaT
    # Prepare sample rows (top by original timestamp descending)
    if sample_cols is None:
        sample_cols = df.columns.tolist()
    # ensure we include the localized ts (string) for readability
    sample_df = df.sort_values('timestamp', ascending=False).head(sample_limit)
    # convert ts to ISO string for compactness
    sample_df = sample_df.loc[:, sample_cols + ['ts']].copy()
    sample_df['ts'] = sample_df['ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    # Build summary row as DataFrame for compact display
    summary = {
        'rows': row_count,
        'distinct_'+id_col+'_count': distinct_ids_count,
        'min_ts_utc+8': (global_min_ts.strftime('%Y-%m-%d %H:%M:%S %Z') if pd.notna(global_min_ts) else None),
        'max_ts_utc+8': (global_max_ts.strftime('%Y-%m-%d %H:%M:%S %Z') if pd.notna(global_max_ts) else None)
    }
    if name_col:
        summary['distinct_'+name_col+'_count'] = distinct_names_count
    summary_df = pd.DataFrame([summary])
    return summary_df, distinct_ids, distinct_names, sample_df

# Summarize metric.csv
metric_summary, metric_cmdbs, metric_kpis, metric_top20 = summarize_file(
    df_metric,
    id_col='cmdb_id',
    name_col='kpi_name',
    sample_cols=['timestamp','cmdb_id','kpi_name','value']
)

# Summarize trace.csv
trace_summary, trace_cmdbs, trace_traces, trace_top20 = summarize_file(
    df_trace,
    id_col='cmdb_id',
    name_col='trace_name',
    sample_cols=['timestamp','cmdb_id','trace_name','value']
)

# Summarize log.csv
log_summary, log_cmdbs, log_logs, log_top20 = summarize_file(
    df_log,
    id_col='cmdb_id',
    name_col='log_name',
    sample_cols=['timestamp','cmdb_id','log_name','value']
)

# Summarize error_logs.csv (no name_col)
error_summary, error_cmdbs, _, error_top20 = summarize_file(
    df_error,
    id_col='cmdb_id',
    name_col=None,
    sample_cols=['timestamp','cmdb_id','message']
)

# Keep outputs compact: show summaries, distinct lists (up to 50), and top-20 samples for each file
metric_summary, metric_cmdbs, metric_kpis, metric_top20, \
trace_summary, trace_cmdbs, trace_traces, trace_top20, \
log_summary, log_cmdbs, log_logs, log_top20, \
error_summary, error_cmdbs, error_top20

```
Out[1]:
```
summary = (
    "Summary of loaded telemetry (UTC+8):\n\n"
    "metric.csv:\n"
    "- Rows: 144,252\n"
    "- Distinct cmdb_id: 59 (examples: adservice, adservice-0, adservice-1, adservice-2, adservice2, "
    "productcatalogservice-2, recommendationservice)\n"
    "- Distinct kpi_name: 1,152\n"
    "- Time range: 2022-03-20 11:00:00 to 2022-03-20 11:29:00 (CST)\n"
    "- Top-20 (latest) samples: many container and mesh metrics at 2022-03-20 11:29:00, e.g., shippingservice2-0 metrics\n\n"
    "trace.csv:\n"
    "- Rows: 36,632\n"
    "- Distinct cmdb_id: 40 (examples: adservice-0, cartservice-0, shippingservice-0/1/2, shippingservice2-0)\n"
    "- Distinct trace_name: 200\n"
    "- Time range: 2022-03-20 11:00:00 to 2022-03-20 11:29:00 (CST)\n"
    "- Top-20 (latest) samples: many trace features at 2022-03-20 11:29:00, showing traces from frontend→services and checkout→shipping\n\n"
    "log.csv:\n"
    "- Rows: 1,814\n"
    "- Distinct cmdb_id: 31 (examples: adservice-0/1/2, cartservice-0/1, frontend-1/2, productcatalogservice-0/1/2)\n"
    "- Distinct log_name: 2 (log.error_count, log.row_count)\n"
    "- Time range: 2022-03-20 11:00:00 to 2022-03-20 11:29:00 (CST)\n"
    "- Top-20 (latest) samples: row_count values high for frontend and productcatalogservice; frontend-2 shows log.error_count = 27 at 11:29:00\n\n"
    "error_logs.csv:\n"
    "- Rows: 1,839\n"
    "- Distinct cmdb_id: 3 (frontend-0, frontend-1, frontend-2)\n"
    "- Time range: 2022-03-20 11:00:01 to 2022-03-20 11:29:59 (CST)\n"
    "- Top-20 (latest) samples: repeated warnings (e.g., 'failed to retrieve...') across frontend pods in the 11:29:40–11:29:59 window\n\n"
    "Overall note: Telemetry is concentrated within a 30-minute window (11:00–11:29 UTC+8 on 2022-03-20). "
    "Metrics and traces show broad coverage across many services/pods; logs are smaller in volume and error_logs are focused on frontend pods."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(     rows  distinct_cmdb_id_count             min_ts_utc+8             max_ts_utc+8  distinct_kpi_name_count
0  144252                      59  2022-03-20 11:00:00 CST  2022-03-20 11:29:00 CST                     1152,                      cmdb_id
0                  adservice
1                adservice-0
2                adservice-1
3                adservice-2
4                 adservice2
..                       ...
45   productcatalogservice-2
46  productcatalogservice2-0
47     recommendationservice
48   recommendationservice-0
49   recommendationservice-1

[50 rows x 1 columns],                                              kpi_name
0                                      app.grpc.count
1                                        app.grpc.mrt
2                                         app.grpc.rr
3                                         app.grpc.sr
4                                      app.http.count
..                                                ...
45             container.node-5.container_memory_swap
46         container.node-5.container_memory_usage_MB
47   container.node-5.container_memory_working_set_MB
48  container.node-5.container_network_receive_MB....
49  container.node-5.container_network_receive_err...

[50 rows x 1 columns],          timestamp             cmdb_id                                           kpi_name         value                       ts
144219  1647746940  shippingservice2-0  container.node-5.container_network_transmit_pa...      0.000000  2022-03-20 11:29:00 CST
144190  1647746940  shippingservice2-0  container.node-5.container_fs_reads_merged./de...      0.000000  2022-03-20 11:29:00 CST
144189  1647746940  shippingservice2-0    container.node-5.container_fs_reads_MB./dev/vda      0.000000  2022-03-20 11:29:00 CST
144213  1647746940  shippingservice2-0  container.node-5.container_network_receive_err...      0.000000  2022-03-20 11:29:00 CST
144251  1647746940  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....   1940.250000  2022-03-20 11:29:00 CST
144250  1647746940  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....     18.500000  2022-03-20 11:29:00 CST
144249  1647746940  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....      0.008050  2022-03-20 11:29:00 CST
144248  1647746940  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....  60225.000000  2022-03-20 11:29:00 CST
144247  1647746940  shippingservice2-0  mesh.destination.frontend2.shippingservice2.is...     18.500000  2022-03-20 11:29:00 CST
144246  1647746940  shippingservice2-0  mesh.destination.frontend2.shippingservice2.is...  24375.000000  2022-03-20 11:29:00 CST
144245  1647746940  shippingservice2-0  mesh.destination.frontend2.shippingservice2.is...     18.500000  2022-03-20 11:29:00 CST
144244  1647746940  shippingservice2-0  mesh.destination.frontend2.shippingservice2.is...     18.500000  2022-03-20 11:29:00 CST
144243  1647746940  shippingservice2-0  mesh.destination.frontend2.shippingservice2.is...      0.000525  2022-03-20 11:29:00 CST
144242  1647746940  shippingservice2-0  mesh.destination.frontend2.shippingservice2.is...  12702.500000  2022-03-20 11:29:00 CST
144241  1647746940  shippingservice2-0  mesh.destination.checkoutservice2.shippingserv...      9.000000  2022-03-20 11:29:00 CST
144240  1647746940  shippingservice2-0  mesh.destination.checkoutservice2.shippingserv...  11250.000000  2022-03-20 11:29:00 CST
144239  1647746940  shippingservice2-0  mesh.destination.checkoutservice2.shippingserv...      9.000000  2022-03-20 11:29:00 CST
144238  1647746940  shippingservice2-0  mesh.destination.checkoutservice2.shippingserv...      9.000000  2022-03-20 11:29:00 CST
144230  1647746940  shippingservice2-0    container.node-5.container_tasks_state.sleeping      0.000000  2022-03-20 11:29:00 CST
144231  1647746940  shippingservice2-0     container.node-5.container_tasks_state.stopped      0.000000  2022-03-20 11:29:00 CST,     rows  distinct_cmdb_id_count             min_ts_utc+8             max_ts_utc+8  distinct_trace_name_count
0  36632                      40  2022-03-20 11:00:00 CST  2022-03-20 11:29:00 CST                        200,                      cmdb_id
0                adservice-0
1                adservice-1
2                adservice-2
3               adservice2-0
4              cartservice-0
..                       ...
35  recommendationservice2-0
36         shippingservice-0
37         shippingservice-1
38         shippingservice-2
39        shippingservice2-0

[40 rows x 1 columns],                                           trace_name
0         trace.from_checkoutservice-0.duration_mean
1          trace.from_checkoutservice-0.duration_p95
2            trace.from_checkoutservice-0.error_rate
3             trace.from_checkoutservice-0.row_count
4         trace.from_checkoutservice-1.duration_mean
..                                               ...
45  trace.from_recommendationservice2-0.duration_p95
46    trace.from_recommendationservice2-0.error_rate
47     trace.from_recommendationservice2-0.row_count
48                     trace.from_root.duration_mean
49                      trace.from_root.duration_p95

[50 rows x 1 columns],         timestamp             cmdb_id                                  trace_name     value                       ts
36603  1647746940   shippingservice-2      trace.from_checkoutservice-0.row_count  3.000000  2022-03-20 11:29:00 CST
36582  1647746940   shippingservice-1     trace.from_checkoutservice-1.error_rate  0.000000  2022-03-20 11:29:00 CST
36575  1647746940   shippingservice-0             trace.from_frontend-2.row_count  3.000000  2022-03-20 11:29:00 CST
36595  1647746940   shippingservice-1             trace.from_frontend-1.row_count  8.000000  2022-03-20 11:29:00 CST
36631  1647746940  shippingservice2-0            trace.from_frontend2-0.row_count  4.000000  2022-03-20 11:29:00 CST
36630  1647746940  shippingservice2-0           trace.from_frontend2-0.error_rate  0.000000  2022-03-20 11:29:00 CST
36629  1647746940  shippingservice2-0         trace.from_frontend2-0.duration_p95  0.000058  2022-03-20 11:29:00 CST
36628  1647746940  shippingservice2-0        trace.from_frontend2-0.duration_mean  0.000057  2022-03-20 11:29:00 CST
36576  1647746940   shippingservice-1  trace.from_checkoutservice-0.duration_mean  0.000057  2022-03-20 11:29:00 CST
36577  1647746940   shippingservice-1   trace.from_checkoutservice-0.duration_p95  0.000058  2022-03-20 11:29:00 CST
36578  1647746940   shippingservice-1     trace.from_checkoutservice-0.error_rate  0.000000  2022-03-20 11:29:00 CST
36579  1647746940   shippingservice-1      trace.from_checkoutservice-0.row_count  2.000000  2022-03-20 11:29:00 CST
36564  1647746940   shippingservice-0         trace.from_frontend-0.duration_mean  0.000061  2022-03-20 11:29:00 CST
36565  1647746940   shippingservice-0          trace.from_frontend-0.duration_p95  0.000067  2022-03-20 11:29:00 CST
36566  1647746940   shippingservice-0            trace.from_frontend-0.error_rate  0.000000  2022-03-20 11:29:00 CST
36567  1647746940   shippingservice-0             trace.from_frontend-0.row_count  6.000000  2022-03-20 11:29:00 CST
36568  1647746940   shippingservice-0         trace.from_frontend-1.duration_mean  0.000059  2022-03-20 11:29:00 CST
36569  1647746940   shippingservice-0          trace.from_frontend-1.duration_p95  0.000068  2022-03-20 11:29:00 CST
36570  1647746940   shippingservice-0            trace.from_frontend-1.error_rate  0.000000  2022-03-20 11:29:00 CST
36571  1647746940   shippingservice-0             trace.from_frontend-1.row_count  8.000000  2022-03-20 11:29:00 CST,    rows  distinct_cmdb_id_count             min_ts_utc+8             max_ts_utc+8  distinct_log_name_count
0  1814                      31  2022-03-20 11:00:00 CST  2022-03-20 11:29:00 CST                        2,                     cmdb_id
0               adservice-0
1               adservice-1
2               adservice-2
3             cartservice-0
4             cartservice-1
..                      ...
26  recommendationservice-2
27             redis-cart-0
28        shippingservice-0
29        shippingservice-1
30        shippingservice-2

[31 rows x 1 columns],           log_name
0  log.error_count
1    log.row_count,        timestamp                  cmdb_id         log_name   value                       ts
1755  1647746940              adservice-0    log.row_count  1400.0  2022-03-20 11:29:00 CST
1778  1647746940           emailservice-0  log.error_count     0.0  2022-03-20 11:29:00 CST
1813  1647746940        shippingservice-2    log.row_count    81.0  2022-03-20 11:29:00 CST
1812  1647746940        shippingservice-2  log.error_count     0.0  2022-03-20 11:29:00 CST
1811  1647746940        shippingservice-1    log.row_count    88.0  2022-03-20 11:29:00 CST
1810  1647746940        shippingservice-1  log.error_count     0.0  2022-03-20 11:29:00 CST
1809  1647746940        shippingservice-0    log.row_count    86.0  2022-03-20 11:29:00 CST
1808  1647746940        shippingservice-0  log.error_count     0.0  2022-03-20 11:29:00 CST
1795  1647746940         paymentservice-2    log.row_count    12.0  2022-03-20 11:29:00 CST
1796  1647746940  productcatalogservice-0  log.error_count     0.0  2022-03-20 11:29:00 CST
1797  1647746940  productcatalogservice-0    log.row_count  1076.0  2022-03-20 11:29:00 CST
1798  1647746940  productcatalogservice-1  log.error_count     0.0  2022-03-20 11:29:00 CST
1799  1647746940  productcatalogservice-1    log.row_count  1078.0  2022-03-20 11:29:00 CST
1800  1647746940  productcatalogservice-2  log.error_count     0.0  2022-03-20 11:29:00 CST
1801  1647746940  productcatalogservice-2    log.row_count  1091.0  2022-03-20 11:29:00 CST
1802  1647746940  recommendationservice-0  log.error_count     0.0  2022-03-20 11:29:00 CST
1787  1647746940               frontend-1    log.row_count  1426.0  2022-03-20 11:29:00 CST
1788  1647746940               frontend-2  log.error_count    27.0  2022-03-20 11:29:00 CST
1789  1647746940               frontend-2    log.row_count  1590.0  2022-03-20 11:29:00 CST
1790  1647746940         paymentservice-0  log.error_count     0.0  2022-03-20 11:29:00 CST,    rows  distinct_cmdb_id_count             min_ts_utc+8             max_ts_utc+8
0  1839                       3  2022-03-20 11:00:01 CST  2022-03-20 11:29:59 CST,       cmdb_id
0  frontend-0
1  frontend-1
2  frontend-2,        timestamp     cmdb_id                                            message                       ts
1838  1647746999  frontend-2  severity: warning, message: failed to retrieve...  2022-03-20 11:29:59 CST
1837  1647746997  frontend-0  severity: warning, message: failed to retrieve...  2022-03-20 11:29:57 CST
1836  1647746997  frontend-2  severity: warning, message: failed to retrieve...  2022-03-20 11:29:57 CST
1835  1647746995  frontend-0  severity: warning, message: failed to retrieve...  2022-03-20 11:29:55 CST
1834  1647746995  frontend-1  severity: warning, message: failed to retrieve...  2022-03-20 11:29:55 CST
1833  1647746994  frontend-2  severity: warning, message: failed to retrieve...  2022-03-20 11:29:54 CST
1832  1647746993  frontend-2  severity: warning, message: failed to retrieve...  2022-03-20 11:29:53 CST
1831  1647746992  frontend-1  severity: warning, message: failed to retrieve...  2022-03-20 11:29:52 CST
1830  1647746991  frontend-0  severity: warning, message: failed to retrieve...  2022-03-20 11:29:51 CST
1829  1647746990  frontend-1  severity: warning, message: failed to retrieve...  2022-03-20 11:29:50 CST
1828  1647746989  frontend-0  severity: warning, message: failed to retrieve...  2022-03-20 11:29:49 CST
1827  1647746988  frontend-2  severity: warning, message: failed to retrieve...  2022-03-20 11:29:48 CST
1826  1647746986  frontend-2  severity: warning, message: failed to retrieve...  2022-03-20 11:29:46 CST
1824  1647746985  frontend-2  severity: warning, message: failed to retrieve...  2022-03-20 11:29:45 CST
1825  1647746985  frontend-0  severity: warning, message: failed to retrieve...  2022-03-20 11:29:45 CST
1822  1647746982  frontend-1  severity: warning, message: failed to retrieve...  2022-03-20 11:29:42 CST
1823  1647746982  frontend-2  severity: warning, message: failed to retrieve...  2022-03-20 11:29:42 CST
1821  1647746981  frontend-0  severity: warning, message: failed to retrieve...  2022-03-20 11:29:41 CST
1819  1647746980  frontend-0  severity: warning, message: failed to retrieve...  2022-03-20 11:29:40 CST
1820  1647746980  frontend-1  severity: warning, message: failed to retrieve...  2022-03-20 11:29:40 CST)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz

# reuse existing df_metric and tz if present in kernel; otherwise load metric.csv
try:
    df_metric
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')
    df_metric = pd.read_csv('metric.csv')
    df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Ensure value is numeric and ts exists
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
if 'ts' not in df_metric.columns:
    df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Step 1: compute global P95, P5, and total points per (cmdb_id, kpi_name) using full series
grp = df_metric.groupby(['cmdb_id', 'kpi_name'])['value']
thresholds = grp.agg(
    total_points_full_series = 'count',
    P95 = lambda x: x.quantile(0.95),
    P5  = lambda x: x.quantile(0.05)
).reset_index()

# Step 2: filter to UTC+8 window 2022-03-20 11:00:00 through 2022-03-20 11:30:00 (inclusive)
start = pd.Timestamp('2022-03-20 11:00:00', tz=tz)
end   = pd.Timestamp('2022-03-20 11:30:00', tz=tz)
df_window = df_metric[(df_metric['ts'] >= start) & (df_metric['ts'] <= end)].copy()

# compute window-level stats: points, median, IQR
win_grp = df_window.groupby(['cmdb_id', 'kpi_name'])['value']
window_stats = win_grp.agg(
    points_in_window = 'count',
    median_in_window = 'median',
    iqr_in_window = lambda x: x.quantile(0.75) - x.quantile(0.25)
).reset_index()

# join thresholds to window rows to detect anomalies
df_win_with_thresh = df_window.merge(thresholds[['cmdb_id','kpi_name','P95','P5']], on=['cmdb_id','kpi_name'], how='left')
df_win_with_thresh['is_anomaly'] = ((df_win_with_thresh['value'] >= df_win_with_thresh['P95']) |
                                    (df_win_with_thresh['value'] <= df_win_with_thresh['P5']))

# aggregate anomaly counts and earliest anomaly ts in window
anom_agg = df_win_with_thresh[df_win_with_thresh['is_anomaly']].groupby(['cmdb_id','kpi_name']).agg(
    anomaly_count_in_window = ('is_anomaly','sum'),
    earliest_anomaly_ts = ('ts','min')
).reset_index()

# merge everything: thresholds + window_stats + anomaly aggregates
summary = thresholds.merge(window_stats, on=['cmdb_id','kpi_name'], how='left')
summary = summary.merge(anom_agg, on=['cmdb_id','kpi_name'], how='left')

# fill NaNs for counts and format earliest timestamp
summary['points_in_window'] = summary['points_in_window'].fillna(0).astype(int)
summary['anomaly_count_in_window'] = summary['anomaly_count_in_window'].fillna(0).astype(int)
summary['earliest_anomaly_timestamp_in_window'] = summary['earliest_anomaly_ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
summary.drop(columns=['earliest_anomaly_ts'], inplace=True)

# Rename total_points_full_series column if needed
if 'total_points_full_series' not in summary.columns and 'total_points_full_series' in summary:
    pass  # already named
# For compactness select needed columns and sort by anomaly count desc
result = summary[['cmdb_id','kpi_name','anomaly_count_in_window','earliest_anomaly_timestamp_in_window','P95','P5','points_in_window','total_points_full_series']]
top20 = result.sort_values('anomaly_count_in_window', ascending=False).head(20).reset_index(drop=True)

# Step 4: fallback if none exceed 0 anomalies -> rank by max deviation within window
if top20['anomaly_count_in_window'].max() == 0:
    # compute per-row normalized deviation = abs(value - median)/IQR (use window median/iqr)
    # join median and iqr to window rows
    df_dev = df_window.merge(window_stats[['cmdb_id','kpi_name','median_in_window','iqr_in_window']], on=['cmdb_id','kpi_name'], how='left')
    # compute deviation, handle iqr==0 by setting deviation = NaN (will be treated as 0 later)
    df_dev['dev_norm'] = np.where(df_dev['iqr_in_window']>0,
                                  (df_dev['value'] - df_dev['median_in_window']).abs() / df_dev['iqr_in_window'],
                                  np.nan)
    # per-group max deviation
    max_dev = df_dev.groupby(['cmdb_id','kpi_name'])['dev_norm'].max().reset_index().rename(columns={'dev_norm':'max_dev_norm'})
    # merge into summary and pick top by max_dev_norm
    fallback = summary.merge(max_dev, on=['cmdb_id','kpi_name'], how='left')
    fallback['max_dev_norm'] = fallback['max_dev_norm'].fillna(0)
    # prepare fallback result columns similar to required ones, adding max_dev_norm for context
    fallback_result = fallback[['cmdb_id','kpi_name','anomaly_count_in_window','earliest_anomaly_timestamp_in_window','P95','P5','points_in_window','total_points_full_series','max_dev_norm']]
    top20_fallback = fallback_result.sort_values('max_dev_norm', ascending=False).head(20).reset_index(drop=True)
    final_top20 = top20_fallback
else:
    final_top20 = top20

# Keep output compact: show shape and the top rows (<=20) with selected columns
final_top20.shape, final_top20.head(20)

```
Out[2]:
```
summary = (
    "Summary of metric anomaly analysis (window 2022-03-20 11:00–11:30 UTC+8):\n\n"
    "- Top findings: The top 20 (cmdb_id, kpi_name) pairs with the most anomalies are all for cmdb_id = shippingservice2-0.\n"
    "- For each of these 20 series:\n"
    "  * anomaly_count_in_window = 30 (i.e., every point in the 30-minute window was flagged as anomalous)\n"
    "  * points_in_window = 30 and total_points_full_series = 30 (the full series covers only this window)\n"
    "  * earliest anomaly timestamp = 2022-03-20 11:00:00 (UTC+8)\n"
    "  * KPIs include many container.node-5.* metrics (fs_reads/writes, memory, ulimits, fs_io, etc.) and some mesh.destination.* metrics.\n\n"
    "- Implication: shippingservice2-0 (node-5 container metrics) consistently exceeded the global P95 or fell below P5 for the entire 11:00–11:30 window, so these series are persistently outside their usual range during this period.\n\n"
    "Recommendation: investigate shippingservice2-0 (node-5) container filesystem/memory/ulimits and related mesh traffic for root causes (resource limits, disk/io, or container configuration)."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

((20, 8),                cmdb_id                                           kpi_name  anomaly_count_in_window earliest_anomaly_timestamp_in_window           P95         P5  points_in_window  total_points_full_series
0   shippingservice2-0  mesh.destination.checkoutservice2.shippingserv...                       30              2022-03-20 11:00:00 CST  2.887500e-04        0.0                30                        30
1   shippingservice2-0  container.node-5.container_ulimits_soft.max_op...                       30              2022-03-20 11:00:00 CST  6.553600e+04    65536.0                30                        30
2   shippingservice2-0     container.node-5.container_fs_writes./dev/vda1                       30              2022-03-20 11:00:00 CST  0.000000e+00        0.0                30                        30
3   shippingservice2-0   container.node-5.container_fs_writes_MB./dev/vda                       30              2022-03-20 11:00:00 CST  0.000000e+00        0.0                30                        30
4   shippingservice2-0  container.node-5.container_fs_writes_merged./d...                       30              2022-03-20 11:00:00 CST  0.000000e+00        0.0                30                        30
5   shippingservice2-0            container.node-5.container_memory_cache                       30              2022-03-20 11:00:00 CST  9.650176e+06  9650176.0                30                        30
6   shippingservice2-0          container.node-5.container_memory_failcnt                       30              2022-03-20 11:00:00 CST  0.000000e+00        0.0                30                        30
7   shippingservice2-0  container.node-5.container_memory_failures.con...                       30              2022-03-20 11:00:00 CST  0.000000e+00        0.0                30                        30
8   shippingservice2-0      container.node-5.container_fs_reads./dev/vda1                       30              2022-03-20 11:00:00 CST  0.000000e+00        0.0                30                        30
9   shippingservice2-0    container.node-5.container_fs_reads_MB./dev/vda                       30              2022-03-20 11:00:00 CST  0.000000e+00        0.0                30                        30
10  shippingservice2-0  container.node-5.container_fs_reads_merged./de...                       30              2022-03-20 11:00:00 CST  0.000000e+00        0.0                30                        30
11  shippingservice2-0  container.node-5.container_fs_sector_reads./de...                       30              2022-03-20 11:00:00 CST  0.000000e+00        0.0                30                        30
12  shippingservice2-0  container.node-5.container_fs_sector_writes./d...                       30              2022-03-20 11:00:00 CST  0.000000e+00        0.0                30                        30
13  shippingservice2-0  container.node-5.container_fs_write_seconds./d...                       30              2022-03-20 11:00:00 CST  0.000000e+00        0.0                30                        30
14  shippingservice2-0      container.node-5.container_fs_writes./dev/vda                       30              2022-03-20 11:00:00 CST  0.000000e+00        0.0                30                        30
15  shippingservice2-0     container.node-5.container_fs_inodes./dev/vda1                       30              2022-03-20 11:00:00 CST  0.000000e+00        0.0                30                        30
16  shippingservice2-0  container.node-5.container_fs_inodes_free./dev...                       30              2022-03-20 11:00:00 CST  0.000000e+00        0.0                30                        30
17  shippingservice2-0  container.node-5.container_fs_io_current./dev/...                       30              2022-03-20 11:00:00 CST  0.000000e+00        0.0                30                        30
18  shippingservice2-0  container.node-5.container_fs_io_time_seconds....                       30              2022-03-20 11:00:00 CST  0.000000e+00        0.0                30                        30
19  shippingservice2-0  container.node-5.container_fs_io_time_weighted...                       30              2022-03-20 11:00:00 CST  0.000000e+00        0.0                30                        30)```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# reuse or load metric data
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

# ensure ts and numeric value
if 'ts' not in df_metric.columns:
    df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

# reuse or compute thresholds (P95/P5) per (cmdb_id, kpi_name)
try:
    thresholds
except NameError:
    grp = df_metric.groupby(['cmdb_id', 'kpi_name'])['value']
    thresholds = grp.agg(
        total_points_full_series = 'count',
        P95 = lambda x: x.quantile(0.95),
        P5  = lambda x: x.quantile(0.05)
    ).reset_index()

# Window filter (inclusive)
start = pd.Timestamp('2022-03-20 11:00:00', tz=tz)
end   = pd.Timestamp('2022-03-20 11:30:00', tz=tz)
df_window = df_metric[(df_metric['ts'] >= start) & (df_metric['ts'] <= end)].copy()

# points in window per series
points_in_window = df_window.groupby(['cmdb_id','kpi_name'])['value'].count().reset_index().rename(columns={'value':'points_in_window'})

# merge thresholds into window rows to evaluate anomalies
df_win_thresh = df_window.merge(thresholds[['cmdb_id','kpi_name','P95','P5']], on=['cmdb_id','kpi_name'], how='left')
df_win_thresh['is_anomaly'] = ((df_win_thresh['value'] >= df_win_thresh['P95']) |
                               (df_win_thresh['value'] <= df_win_thresh['P5']))

# per-series anomaly aggregates: count and earliest anomaly ts
anom_agg = (df_win_thresh[df_win_thresh['is_anomaly']]
            .groupby(['cmdb_id','kpi_name'])
            .agg(anomaly_count_in_window=('is_anomaly','sum'),
                 earliest_anomaly_ts=('ts','min'))
            .reset_index())

# assemble series-level summary
series_summary = thresholds[['cmdb_id','kpi_name','total_points_full_series','P95','P5']].merge(
    points_in_window, on=['cmdb_id','kpi_name'], how='left'
).merge(
    anom_agg, on=['cmdb_id','kpi_name'], how='left'
)

series_summary['points_in_window'] = series_summary['points_in_window'].fillna(0).astype(int)
series_summary['anomaly_count_in_window'] = series_summary['anomaly_count_in_window'].fillna(0).astype(int)

# Aggregate by cmdb_id
comp_agg = series_summary.groupby('cmdb_id').agg(
    total_anomaly_count = ('anomaly_count_in_window','sum'),
    points_in_window_total = ('points_in_window','sum')
).reset_index()

# distinct anomalous kpi count per component
distinct_counts = (series_summary[series_summary['anomaly_count_in_window']>0]
                   .groupby('cmdb_id')['kpi_name'].nunique()
                   .reset_index().rename(columns={'kpi_name':'distinct_anomalous_kpi_count'}))

comp_agg = comp_agg.merge(distinct_counts, on='cmdb_id', how='left')
comp_agg['distinct_anomalous_kpi_count'] = comp_agg['distinct_anomalous_kpi_count'].fillna(0).astype(int)

# earliest anomaly timestamp per component
earliest_comp = (series_summary.dropna(subset=['earliest_anomaly_ts'])
                 .groupby('cmdb_id')['earliest_anomaly_ts']
                 .min()
                 .reset_index())

comp_agg = comp_agg.merge(earliest_comp, on='cmdb_id', how='left')
comp_agg['earliest_anomaly_timestamp_for_component'] = comp_agg['earliest_anomaly_ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
comp_agg.drop(columns=['earliest_anomaly_ts'], inplace=True)

# final ordering and select columns
comp_final = comp_agg[['cmdb_id','total_anomaly_count','distinct_anomalous_kpi_count','earliest_anomaly_timestamp_for_component','points_in_window_total']]
comp_top20 = comp_final.sort_values('total_anomaly_count', ascending=False).head(20).reset_index(drop=True)

# Note if only shippingservice2-0 is anomalous
anomalous_components = comp_top20[comp_top20['total_anomaly_count']>0]['cmdb_id'].unique()
if len(anomalous_components) == 1 and anomalous_components[0] == 'shippingservice2-0':
    note = "Only shippingservice2-0 shows anomalies in the window."
else:
    note = ""

comp_top20, note

```
Out[3]:
```
Summary (top 20 components by anomaly count, window 2022-03-20 11:00–11:30 UTC+8):

- The audit aggregated anomalies across KPIs per component and sorted by total anomaly count. All reported components have their earliest anomaly at 2022-03-20 11:00:00 (UTC+8).
- Top 3 by total_anomaly_count:
  1. adservice — total_anomaly_count 9,054 across 341 distinct KPIs; points_in_window_total 10,230.
  2. adservice2 — total_anomaly_count 8,925 across 333 distinct KPIs; points_in_window_total 9,990.
  3. frontend2-0 — total_anomaly_count 3,052 across 160 distinct KPIs; points_in_window_total 4,800.
- Other notable components in the top 20 include frontend-0, frontend-1, frontend-2, productcatalogservice-0/1/2, checkoutservice-0/1/2, and several cartservice and adservice pods — each with hundreds to a few thousand anomaly points and dozens–hundreds of distinct anomalous KPIs.
- Interpretation: Many services (especially adservice, its replica adservice2, frontends, productcatalogservice and checkoutservice family) show widespread KPI anomalies during the entire 30-minute window (large numbers of KPI-series and data points flagged against their global P95/P5 thresholds). This suggests broad, systemic deviations rather than a single isolated KPI.
- Recommended next steps: prioritize investigation on the highest-ranked components (adservice, adservice2, frontends, productcatalogservice, checkoutservice) — check the specific anomalous KPIs (resource, I/O, mesh/traffic metrics), correlated traces/logs, and recent configuration/deployment changes.

The original code execution output of IPython Kernel is also provided below for reference:

(                     cmdb_id  total_anomaly_count  distinct_anomalous_kpi_count earliest_anomaly_timestamp_for_component  points_in_window_total
0                  adservice                 9054                           341                  2022-03-20 11:00:00 CST                   10230
1                 adservice2                 8925                           333                  2022-03-20 11:00:00 CST                    9990
2                frontend2-0                 3052                           160                  2022-03-20 11:00:00 CST                    4800
3                 frontend-0                 2918                           160                  2022-03-20 11:00:00 CST                    4800
4                 frontend-1                 2822                           156                  2022-03-20 11:00:00 CST                    4680
5                 frontend-2                 2516                           146                  2022-03-20 11:00:00 CST                    4380
6    productcatalogservice-2                 2368                           116                  2022-03-20 11:00:00 CST                    3480
7    productcatalogservice-0                 2325                           116                  2022-03-20 11:00:00 CST                    3480
8    productcatalogservice-1                 2250                           112                  2022-03-20 11:00:00 CST                    3360
9          checkoutservice-0                 2206                           126                  2022-03-20 11:00:00 CST                    3780
10         checkoutservice-1                 2029                           118                  2022-03-20 11:00:00 CST                    3540
11         checkoutservice-2                 2012                           118                  2022-03-20 11:00:00 CST                    3540
12             cartservice-1                 1973                            92                  2022-03-20 11:00:00 CST                    2760
13  productcatalogservice2-0                 1972                            96                  2022-03-20 11:00:00 CST                    2880
14              adservice2-0                 1952                            86                  2022-03-20 11:00:00 CST                    2580
15               adservice-0                 1935                            86                  2022-03-20 11:00:00 CST                    2580
16            cartservice2-0                 1905                            88                  2022-03-20 11:00:00 CST                    2640
17               adservice-1                 1864                            82                  2022-03-20 11:00:00 CST                    2460
18             cartservice-0                 1853                            88                  2022-03-20 11:00:00 CST                    2640
19        checkoutservice2-0                 1853                           110                  2022-03-20 11:00:00 CST                    3300, '')```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

# reuse existing df_trace and tz if available; otherwise load
try:
    df_trace
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')
    df_trace = pd.read_csv('trace.csv')
    df_trace['ts'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# ensure ts and numeric value
if 'ts' not in df_trace.columns:
    df_trace['ts'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')

# Step 1: global P95 and P5 per (cmdb_id, trace_name) using full series
trace_grp = df_trace.groupby(['cmdb_id','trace_name'])['value']
trace_thresholds = trace_grp.agg(
    total_points_full_series = 'count',
    P95 = lambda x: x.quantile(0.95),
    P5  = lambda x: x.quantile(0.05)
).reset_index()

# Step 2: filter to window 2022-03-20 11:00:00 through 2022-03-20 11:30:00 (UTC+8)
start = pd.Timestamp('2022-03-20 11:00:00', tz=tz)
end   = pd.Timestamp('2022-03-20 11:30:00', tz=tz)
df_twin = df_trace[(df_trace['ts'] >= start) & (df_trace['ts'] <= end)].copy()

# Step 3: classify anomaly rule by trace_name
# create type column
def trace_type(name):
    n = str(name).lower()
    if 'error_rate' in n:
        return 'error_rate'
    if 'duration' in n:
        return 'duration'
    if 'row_count' in n:
        return 'row_count'
    return 'other'

df_twin['trace_type'] = df_twin['trace_name'].map(trace_type)

# merge thresholds into window rows
df_twin = df_twin.merge(trace_thresholds[['cmdb_id','trace_name','P95','P5']], on=['cmdb_id','trace_name'], how='left')

# apply anomaly logic:
# - error_rate or duration: anomaly if value >= P95
# - row_count: anomaly if value <= P5
# - other: mark False
cond_ge_p95 = df_twin['value'] >= df_twin['P95']
cond_le_p5  = df_twin['value'] <= df_twin['P5']

df_twin['is_anomaly'] = False
df_twin.loc[df_twin['trace_type'].isin(['error_rate','duration']), 'is_anomaly'] = cond_ge_p95[df_twin['trace_type'].isin(['error_rate','duration'])]
df_twin.loc[df_twin['trace_type']=='row_count', 'is_anomaly'] = cond_le_p5[df_twin['trace_type']=='row_count']

# Step 4: per (cmdb_id, trace_name) compute anomaly_count_in_window, earliest_anomaly_timestamp_in_window, points_in_window
points = df_twin.groupby(['cmdb_id','trace_name'])['value'].count().reset_index().rename(columns={'value':'points_in_window'})

anom_grp = (df_twin[df_twin['is_anomaly']]
            .groupby(['cmdb_id','trace_name'])
            .agg(anomaly_count_in_window=('is_anomaly','sum'),
                 earliest_anomaly_ts=('ts','min'))
            .reset_index())

trace_summary = trace_thresholds[['cmdb_id','trace_name','P95','P5','total_points_full_series']].merge(
    points, on=['cmdb_id','trace_name'], how='left'
).merge(
    anom_grp, on=['cmdb_id','trace_name'], how='left'
)

trace_summary['points_in_window'] = trace_summary['points_in_window'].fillna(0).astype(int)
trace_summary['anomaly_count_in_window'] = trace_summary['anomaly_count_in_window'].fillna(0).astype(int)
trace_summary['earliest_anomaly_timestamp_in_window'] = trace_summary['earliest_anomaly_ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
trace_summary.drop(columns=['earliest_anomaly_ts'], inplace=True)

# Step 5: aggregate by cmdb_id
comp_trace_agg = trace_summary.groupby('cmdb_id').agg(
    total_trace_anomaly_count = ('anomaly_count_in_window','sum'),
    points_in_window_total = ('points_in_window','sum')
).reset_index()

distinct_anomalous_trace_count = (trace_summary[trace_summary['anomaly_count_in_window']>0]
                                  .groupby('cmdb_id')['trace_name'].nunique()
                                  .reset_index().rename(columns={'trace_name':'distinct_anomalous_trace_count'}))

earliest_trace_anom = (trace_summary.dropna(subset=['earliest_anomaly_timestamp_in_window'])
                       .groupby('cmdb_id')['earliest_anomaly_timestamp_in_window']
                       .min()
                       .reset_index().rename(columns={'earliest_anomaly_timestamp_in_window':'earliest_trace_anomaly_timestamp_for_component'}))

comp_trace_agg = comp_trace_agg.merge(distinct_anomalous_trace_count, on='cmdb_id', how='left')
comp_trace_agg['distinct_anomalous_trace_count'] = comp_trace_agg['distinct_anomalous_trace_count'].fillna(0).astype(int)
comp_trace_agg = comp_trace_agg.merge(earliest_trace_anom, on='cmdb_id', how='left')

# Step 6A: Top 20 (cmdb_id, trace_name) by anomaly_count_in_window
top_trace_pairs = trace_summary.sort_values('anomaly_count_in_window', ascending=False)[
    ['cmdb_id','trace_name','anomaly_count_in_window','earliest_anomaly_timestamp_in_window','P95','P5','points_in_window']
].head(20).reset_index(drop=True)

# Step 6B: Top 20 cmdb_id by total_trace_anomaly_count
top_components = comp_trace_agg.sort_values('total_trace_anomaly_count', ascending=False)[
    ['cmdb_id','total_trace_anomaly_count','distinct_anomalous_trace_count','earliest_trace_anomaly_timestamp_for_component']
].head(20).reset_index(drop=True)

# Step 7: list of cmdb_id whose earliest_trace_anomaly_timestamp_for_component >= '2022-03-20 11:29:00' UTC+8
threshold_time = pd.Timestamp('2022-03-20 11:29:00', tz=tz)
# parse earliest timestamps back to tz-aware timestamps for comparison
def parse_ts(s):
    try:
        return pd.to_datetime(s).tz_localize(None).tz_localize(tz) if pd.notna(s) else pd.NaT
    except Exception:
        # try direct parsing with timezone
        try:
            return pd.to_datetime(s).tz_convert(tz)
        except Exception:
            return pd.NaT

# For safe comparison, convert string timestamps to tz-aware via to_datetime (they are in format '%Y-%m-%d %H:%M:%S %Z')
# Simpler: compare original earliest timestamps by re-deriving from trace_summary grouped data
earliest_ts_df = trace_summary.dropna(subset=['earliest_anomaly_timestamp_in_window']).copy()
# convert 'earliest_anomaly_timestamp_in_window' strings back to tz-aware timestamps (they include CST)
earliest_ts_df['earliest_ts_parsed'] = pd.to_datetime(earliest_ts_df['earliest_anomaly_timestamp_in_window'])
# pandas will treat them as naive; localize to Asia/Shanghai
earliest_ts_df['earliest_ts_parsed'] = earliest_ts_df['earliest_ts_parsed'].dt.tz_localize(tz)

comp_earliest = earliest_ts_df.groupby('cmdb_id')['earliest_ts_parsed'].min().reset_index()
late_components = comp_earliest[comp_earliest['earliest_ts_parsed'] >= threshold_time]['cmdb_id'].head(20).reset_index(drop=True)

# Compact outputs: top_trace_pairs (<=20 rows), top_components (<=20 rows), late_components (<=20)
top_trace_pairs, top_components, late_components

```
Out[5]:
```
summary = (
    "Trace analysis summary (window 2022-03-20 11:00–11:30 UTC+8):\n\n"
    "1) Per-trace anomalies:\n"
    "- The top anomalous trace series are mostly error_rate traces (e.g., trace.from_frontend-*.error_rate, trace.to_* .error_rate) with anomaly_count_in_window = 30 (every point in the 30-min window).\n"
    "- For these error_rate series P95 = 0 and P5 = 0, so any nonzero error_rate is treated as anomalous relative to the historical baseline.\n\n"
    "2) Per-component aggregation:\n"
    "- The components with the largest total trace anomaly counts are: checkoutservice (checkoutservice-0/1/2), frontend pods (frontend-1/0/2, frontend2-0), cartservice, productcatalogservice, recommendationservice, and shippingservice.\n"
    "- These components show hundreds to over a thousand trace anomaly events during the 30-min window.\n"
    "- Earliest trace anomalies for these components occur at 2022-03-20 11:00:00 (UTC+8), i.e., at the start of the window.\n\n"
    "3) Alignment with frontend error burst:\n"
    "- No component had its earliest_trace_anomaly_timestamp_for_component at or after 2022-03-20 11:29:00 UTC+8 (the list is empty), so the trace anomalies reported here mostly began at 11:00 and are not limited to a late 11:29 spike.\n\n"
    "Conclusion and next steps:\n"
    "- There is a broad, early-onset increase in trace error rates across many services starting at 11:00, not just a short burst at 11:29. Because P95=0 for many error_rate metrics, any errors are unusual.\n"
    "- Prioritize investigation of checkoutservice, frontend pods, productcatalogservice and related call paths, and inspect logs/mesh metrics and recent deployments/config changes. Also correlate these trace anomalies with the metric and log anomalies already observed for frontends and adservice components."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(               cmdb_id                                   trace_name  anomaly_count_in_window earliest_anomaly_timestamp_in_window  P95   P5  points_in_window
0          adservice-1             trace.from_frontend-1.error_rate                       30              2022-03-20 11:00:00 CST  0.0  0.0                30
1   shippingservice2-0            trace.from_frontend2-0.error_rate                       30              2022-03-20 11:00:00 CST  0.0  0.0                30
2          adservice-1             trace.from_frontend-0.error_rate                       30              2022-03-20 11:00:00 CST  0.0  0.0                30
3          adservice-2             trace.from_frontend-0.error_rate                       30              2022-03-20 11:00:00 CST  0.0  0.0                30
4          adservice-0             trace.from_frontend-0.error_rate                       30              2022-03-20 11:00:00 CST  0.0  0.0                30
5          adservice-1             trace.from_frontend-2.error_rate                       30              2022-03-20 11:00:00 CST  0.0  0.0                30
6   shippingservice2-0     trace.from_checkoutservice2-0.error_rate                       30              2022-03-20 11:00:00 CST  0.0  0.0                30
7          adservice-2             trace.from_frontend-2.error_rate                       30              2022-03-20 11:00:00 CST  0.0  0.0                30
8          adservice-0             trace.from_frontend-1.error_rate                       30              2022-03-20 11:00:00 CST  0.0  0.0                30
9          adservice-2             trace.from_frontend-1.error_rate                       30              2022-03-20 11:00:00 CST  0.0  0.0                30
10      emailservice-2      trace.from_checkoutservice-2.error_rate                       30              2022-03-20 11:00:00 CST  0.0  0.0                30
11          frontend-0        trace.to_currencyservice-0.error_rate                       30              2022-03-20 11:00:00 CST  0.0  0.0                30
12          frontend-0        trace.to_currencyservice-1.error_rate                       30              2022-03-20 11:00:00 CST  0.0  0.0                30
13     emailservice2-0     trace.from_checkoutservice2-0.error_rate                       30              2022-03-20 11:00:00 CST  0.0  0.0                30
14          frontend-0                   trace.from_root.error_rate                       30              2022-03-20 11:00:00 CST  0.0  0.0                30
15      emailservice-2      trace.from_checkoutservice-1.error_rate                       30              2022-03-20 11:00:00 CST  0.0  0.0                30
16          frontend-0  trace.to_productcatalogservice-0.error_rate                       30              2022-03-20 11:00:00 CST  0.0  0.0                30
17          frontend-0              trace.to_adservice-1.error_rate                       30              2022-03-20 11:00:00 CST  0.0  0.0                30
18          frontend-0              trace.to_adservice-2.error_rate                       30              2022-03-20 11:00:00 CST  0.0  0.0                30
19          frontend-0                        trace.self.error_rate                       30              2022-03-20 11:00:00 CST  0.0  0.0                30,                     cmdb_id  total_trace_anomaly_count  distinct_anomalous_trace_count earliest_trace_anomaly_timestamp_for_component
0         checkoutservice-0                       1165                              92                        2022-03-20 11:00:00 CST
1         checkoutservice-2                       1147                              92                        2022-03-20 11:00:00 CST
2         checkoutservice-1                       1132                              92                        2022-03-20 11:00:00 CST
3                frontend-1                        933                              92                        2022-03-20 11:00:00 CST
4                frontend-0                        895                              92                        2022-03-20 11:00:00 CST
5                frontend-2                        862                              92                        2022-03-20 11:00:00 CST
6             cartservice-2                        499                              28                        2022-03-20 11:00:00 CST
7             cartservice-1                        461                              28                        2022-03-20 11:00:00 CST
8             cartservice-0                        424                              28                        2022-03-20 11:00:00 CST
9   productcatalogservice-0                        391                              40                        2022-03-20 11:00:00 CST
10  productcatalogservice-2                        379                              40                        2022-03-20 11:00:00 CST
11  productcatalogservice-1                        372                              40                        2022-03-20 11:00:00 CST
12              frontend2-0                        359                              36                        2022-03-20 11:00:00 CST
13       checkoutservice2-0                        354                              36                        2022-03-20 11:00:00 CST
14  recommendationservice-0                        267                              28                        2022-03-20 11:00:00 CST
15  recommendationservice-1                        264                              28                        2022-03-20 11:00:00 CST
16  recommendationservice-2                        259                              28                        2022-03-20 11:00:00 CST
17        shippingservice-2                        251                              24                        2022-03-20 11:00:00 CST
18        shippingservice-0                        239                              24                        2022-03-20 11:00:00 CST
19        shippingservice-1                        239                              24                        2022-03-20 11:00:00 CST, Series([], Name: cmdb_id, dtype: object))```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load or reuse dataframes
try:
    df_error
    df_log
    df_metric
except NameError:
    df_error = pd.read_csv('error_logs.csv')
    df_log = pd.read_csv('log.csv')
    df_metric = pd.read_csv('metric.csv')

# Ensure ts columns and numeric values
for df, name in [(df_error, 'error'), (df_log, 'log'), (df_metric, 'metric')]:
    if 'ts' not in df.columns:
        df['ts'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
if 'value' in df_log.columns:
    df_log['value'] = pd.to_numeric(df_log['value'], errors='coerce')
if 'value' in df_metric.columns:
    df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

# Window definition
start = pd.Timestamp('2022-03-20 11:00:00', tz=tz)
end   = pd.Timestamp('2022-03-20 11:30:00', tz=tz)

# --- Compute global thresholds (P95/P5) per (cmdb_id, kpi_name) from full metric series ---
thresholds = df_metric.groupby(['cmdb_id','kpi_name'])['value'].agg(
    total_points_full_series='count',
    P95=lambda x: x.quantile(0.95),
    P5 = lambda x: x.quantile(0.05)
).reset_index()

# --- Windowed metric data and anomaly flags ---
df_window = df_metric[(df_metric['ts'] >= start) & (df_metric['ts'] <= end)].copy()
df_win_thresh = df_window.merge(thresholds[['cmdb_id','kpi_name','P95','P5']], on=['cmdb_id','kpi_name'], how='left')

df_win_thresh['is_anomaly'] = ((df_win_thresh['value'] >= df_win_thresh['P95']) | (df_win_thresh['value'] <= df_win_thresh['P5']))

# Per-series window aggregates
points = df_win_thresh.groupby(['cmdb_id','kpi_name'])['value'].count().reset_index().rename(columns={'value':'points_in_window'})
anom_agg = df_win_thresh[df_win_thresh['is_anomaly']].groupby(['cmdb_id','kpi_name']).agg(
    anomaly_count_in_window=('is_anomaly','sum'),
    earliest_anom_ts=('ts','min')
).reset_index()

if not anom_agg.empty:
    anom_agg['earliest_anomaly_timestamp_in_window'] = anom_agg['earliest_anom_ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
else:
    anom_agg['earliest_anomaly_timestamp_in_window'] = pd.Series(dtype='object')

series_summary = thresholds.merge(points, on=['cmdb_id','kpi_name'], how='left').merge(
    anom_agg[['cmdb_id','kpi_name','anomaly_count_in_window','earliest_anomaly_timestamp_in_window']],
    on=['cmdb_id','kpi_name'], how='left'
)
series_summary['points_in_window'] = series_summary['points_in_window'].fillna(0).astype(int)
series_summary['anomaly_count_in_window'] = series_summary['anomaly_count_in_window'].fillna(0).astype(int)
if 'earliest_anomaly_timestamp_in_window' not in series_summary.columns:
    series_summary['earliest_anomaly_timestamp_in_window'] = pd.NA

# --- 1) error_logs for frontends ---
frontends = ['frontend-0','frontend-1','frontend-2','frontend2-0']
err_front_window = df_error[df_error['cmdb_id'].isin(frontends) & (df_error['ts'] >= start) & (df_error['ts'] <= end)].copy()

def short_msg(s, maxlen=160):
    if pd.isna(s):
        return ''
    s = str(s).strip()
    return s if len(s) <= maxlen else s[:maxlen-3] + '...'

err_front_summary_rows = []
for cm in frontends:
    sub = err_front_window[err_front_window['cmdb_id']==cm]
    if sub.empty:
        continue
    earliest = sub['ts'].min().strftime('%Y-%m-%d %H:%M:%S %Z')
    latest   = sub['ts'].max().strftime('%Y-%m-%d %H:%M:%S %Z')
    total_count = int(sub.shape[0])
    samples = sub['message'].dropna().astype(str).drop_duplicates().head(5).apply(short_msg).tolist()
    samples_joined = ' || '.join(samples)
    err_front_summary_rows.append((cm, earliest, latest, total_count, samples_joined))

err_frontend_summary = pd.DataFrame(err_front_summary_rows, columns=['cmdb_id','earliest_timestamp','latest_timestamp','total_message_count','message_samples'])

# per-minute counts for 11:25 through 11:30 aggregated across frontends
minute_start = pd.Timestamp('2022-03-20 11:25:00', tz=tz)
minute_end = pd.Timestamp('2022-03-20 11:30:00', tz=tz)
minutes = pd.date_range(start=minute_start, end=minute_end, freq='T', tz=tz)
if not err_front_window.empty:
    err_front_window['minute'] = err_front_window['ts'].dt.floor('T')
    per_min = err_front_window[err_front_window['minute'].between(minute_start, minute_end)].groupby('minute').size().reset_index(name='count')
else:
    per_min = pd.DataFrame(columns=['minute','count'])
per_min_counts = pd.DataFrame({'minute': minutes}).merge(per_min, on='minute', how='left').fillna(0)
per_min_counts['count'] = per_min_counts['count'].astype(int)
per_min_counts['timestamp'] = per_min_counts['minute'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
error_frontend_minute_counts = per_min_counts[['timestamp','count']]

# --- 2) log.csv (log.error_count) for frontends in window ---
log_errors_frontends = df_log[
    (df_log['log_name']=='log.error_count') &
    (df_log['cmdb_id'].isin(frontends)) &
    (df_log['ts'] >= start) & (df_log['ts'] <= end)
].copy()

if not log_errors_frontends.empty:
    log_errors_frontends['timestamp'] = log_errors_frontends['ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    log_errors_frontends = log_errors_frontends.sort_values('ts')[['timestamp','cmdb_id','value']].head(20)
else:
    log_errors_frontends = pd.DataFrame(columns=['timestamp','cmdb_id','value'])

# --- 3) metric anomalies for adservice*, and frontend2-0 ---
mask = series_summary['cmdb_id'].str.startswith('adservice') | (series_summary['cmdb_id']=='frontend2-0')
adsvc_anoms = series_summary[mask].copy()
top_metric_adservice = adsvc_anoms.sort_values('anomaly_count_in_window', ascending=False)[
    ['cmdb_id','kpi_name','anomaly_count_in_window','earliest_anomaly_timestamp_in_window','P95','P5']
].head(20).reset_index(drop=True)

# --- 4) Node-level aggregation for container.node-1 .. node-6 ---
node_rows = []
for i in range(1,7):
    tag = f'container.node-{i}'
    sel = series_summary[series_summary['kpi_name'].str.contains(tag, na=False)]
    if sel.empty:
        continue
    total_anoms = int(sel['anomaly_count_in_window'].sum())
    distinct_series = int(sel[sel['anomaly_count_in_window']>0]['kpi_name'].nunique())
    earliest_vals = sel[sel['anomaly_count_in_window']>0]['earliest_anomaly_timestamp_in_window'].dropna()
    earliest_ts = earliest_vals.min() if not earliest_vals.empty else pd.NA
    node_rows.append((f'node-{i}', total_anoms, distinct_series, earliest_ts))

node_agg = pd.DataFrame(node_rows, columns=['node','total_anomaly_count_in_window','distinct_anomalous_series_count','earliest_anomaly_timestamp']).sort_values('total_anomaly_count_in_window', ascending=False).reset_index(drop=True)

# Return compact outputs (max 20 rows each)
err_frontend_summary, error_frontend_minute_counts.head(20), log_errors_frontends, top_metric_adservice, node_agg.head(20)

```
Out[6]:
```
Summary of focused analysis (window 2022-03-20 11:00–11:30 UTC+8):

1) error_logs for frontends (frontend-0, frontend-1, frontend-2, frontend2-0)
- Per-pod overview (within window):
  - frontend-0: earliest 11:00:03, latest 11:29:57, total messages 609. Sample messages: "severity: warning, message: failed to retrieve..."
  - frontend-1: earliest 11:00:04, latest 11:29:55, total messages 543. Same sample text.
  - frontend-2: earliest 11:00:01, latest 11:29:59, total messages 687. Same sample text.
- Per-minute aggregated counts across the frontends (11:25 → 11:30):
  - 11:25:00 — 66
  - 11:26:00 — 57
  - 11:27:00 — 61
  - 11:28:00 — 53
  - 11:29:00 — 63
  - 11:30:00 — 0
- Interpretation: frontends produced sustained warning messages ("failed to retrieve...") throughout the window; counts remain elevated through 11:29, with no single isolated huge spike at 11:29 (there is sustained high activity across minutes 11:25–11:29).

2) log.error_count for frontends (sample rows, ascending time)
- Per-minute/error_count samples show error_count values present from 11:00 onward for each frontend pod (examples: frontend-0 24 @11:00, frontend-1 18 @11:00, frontend-2 21 @11:00, and similar values each minute).
- Interpretation: error_count was nonzero throughout the period; the frontend error condition is persistent across the whole 30-minute window (not only a late single spike).

3) Metric anomalies for adservice / adservice2 and frontend2-0
- Top anomalous series (by anomaly count) include mesh/istio request/response metrics for frontend2-0 and multiple app/HTTP/gRPC KPIs for adservice/adservice2 (e.g., app.http.count, app.http.rr, app.http.sr, app.grpc.rr, JVM runtime metric).
- Most of these series have anomaly_count_in_window = 30 and earliest anomaly = 11:00:00, meaning they were flagged as anomalous for essentially the whole window.
- Interpretation: adservice and frontend2-0 show broad KPI anomalies across traffic and runtime metrics starting at 11:00 (not limited to a 11:29-only event).

4) Node-level aggregation (container.node-1 .. node-6)
- node-5: total_anomaly_count_in_window = 51,261 across ~64 distinct series; earliest anomaly 11:00:00.
- node-6: total_anomaly_count_in_window = 10,792 across ~64 distinct series; earliest anomaly 11:00:00.
- Other nodes had zero (or negligible) anomalies in this aggregation.
- Interpretation: node-5 is the dominant source of metric anomalies (by a large margin), with node-6 also affected but much less so.

Conclusions and recommended next steps
- The systemic anomaly begins at 11:00 and affects many services and traces starting at that time. Metrics/trace anomalies and frontend warnings are widespread and persistent throughout the 30-minute window.
- Node-level signal points strongly to node-5 as the primary locus of anomalies (very large anomaly counts across many container KPIs referencing container.node-5). This suggests a node-level issue (node-5) rather than a single isolated service.
- The frontend error burst at ~11:29 is part of a sustained error period — frontend error messages persist through 11:29:59 and per-minute counts remain high in 11:25–11:29. The message text is consistently "severity: warning, message: failed to retrieve..." — inspect those logs for the exact failure context and stack/trace if available.
- Actionable checks:
  - Investigate node-5 health: disk I/O, filesystem metrics, memory, ulimits, container runtime errors, and node-level events around 11:00.
  - Correlate node-5 findings with adservice and frontend2-0 anomalies (mesh traffic and app-level errors).
  - Inspect full frontend error log entries around 11:29 (and earlier) to find exact failure causes and any upstream timeouts or resource errors.

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id       earliest_timestamp         latest_timestamp  total_message_count                                    message_samples
0  frontend-0  2022-03-20 11:00:03 CST  2022-03-20 11:29:57 CST                  609  severity: warning, message: failed to retrieve...
1  frontend-1  2022-03-20 11:00:04 CST  2022-03-20 11:29:55 CST                  543  severity: warning, message: failed to retrieve...
2  frontend-2  2022-03-20 11:00:01 CST  2022-03-20 11:29:59 CST                  687  severity: warning, message: failed to retrieve...,                  timestamp  count
0  2022-03-20 11:25:00 CST     66
1  2022-03-20 11:26:00 CST     57
2  2022-03-20 11:27:00 CST     61
3  2022-03-20 11:28:00 CST     53
4  2022-03-20 11:29:00 CST     63
5  2022-03-20 11:30:00 CST      0,                    timestamp     cmdb_id  value
30   2022-03-20 11:00:00 CST  frontend-0   24.0
32   2022-03-20 11:00:00 CST  frontend-1   18.0
34   2022-03-20 11:00:00 CST  frontend-2   21.0
90   2022-03-20 11:01:00 CST  frontend-0   19.0
92   2022-03-20 11:01:00 CST  frontend-1   16.0
94   2022-03-20 11:01:00 CST  frontend-2   26.0
150  2022-03-20 11:02:00 CST  frontend-0   23.0
152  2022-03-20 11:02:00 CST  frontend-1   18.0
154  2022-03-20 11:02:00 CST  frontend-2   19.0
214  2022-03-20 11:03:00 CST  frontend-2   23.0
212  2022-03-20 11:03:00 CST  frontend-1   18.0
210  2022-03-20 11:03:00 CST  frontend-0   18.0
272  2022-03-20 11:04:00 CST  frontend-0   21.0
274  2022-03-20 11:04:00 CST  frontend-1   17.0
276  2022-03-20 11:04:00 CST  frontend-2   23.0
332  2022-03-20 11:05:00 CST  frontend-0   18.0
334  2022-03-20 11:05:00 CST  frontend-1   18.0
336  2022-03-20 11:05:00 CST  frontend-2   25.0
392  2022-03-20 11:06:00 CST  frontend-0   22.0
394  2022-03-20 11:06:00 CST  frontend-1   19.0,         cmdb_id                                           kpi_name  anomaly_count_in_window earliest_anomaly_timestamp_in_window     P95      P5
0   frontend2-0  mesh.source.frontend2.unknown.istio_response_m...                       30              2022-03-20 11:00:00 CST     0.0     0.0
1   frontend2-0  mesh.source.frontend2.unknown.istio_response_b...                       30              2022-03-20 11:00:00 CST     0.0     0.0
2   frontend2-0  mesh.source.frontend2.unknown.istio_requests.g...                       30              2022-03-20 11:00:00 CST     0.0     0.0
3     adservice                                        app.grpc.rr                       30              2022-03-20 11:00:00 CST   100.0   100.0
4   frontend2-0  mesh.source.frontend2.unknown.istio_request_me...                       30              2022-03-20 11:00:00 CST     0.0     0.0
5     adservice                                     app.http.count                       30              2022-03-20 11:00:00 CST    24.0    24.0
6   frontend2-0  mesh.source.frontend2.unknown.istio_request_du...                       30              2022-03-20 11:00:00 CST     0.0     0.0
7     adservice                                        app.http.rr                       30              2022-03-20 11:00:00 CST   100.0   100.0
8     adservice                                        app.http.sr                       30              2022-03-20 11:00:00 CST   100.0   100.0
9     adservice    runtime.java_lang_ClassLoading_LoadedClassCount                       30              2022-03-20 11:00:00 CST  4990.0  4990.0
10  frontend2-0  mesh.source.frontend2.cartservice2.istio_reque...                       30              2022-03-20 11:00:00 CST     0.0     0.0
11  frontend2-0  mesh.source.frontend2.cartservice2.istio_reque...                       30              2022-03-20 11:00:00 CST     0.0     0.0
12  frontend2-0  mesh.source.frontend2.cartservice2.istio_respo...                       30              2022-03-20 11:00:00 CST     0.0     0.0
13  frontend2-0  mesh.source.frontend2.cartservice2.istio_respo...                       30              2022-03-20 11:00:00 CST     0.0     0.0
14  frontend2-0  mesh.source.frontend2.adservice2.istio_respons...                       30              2022-03-20 11:00:00 CST     0.0     0.0
15  frontend2-0  mesh.source.frontend2.adservice2.istio_respons...                       30              2022-03-20 11:00:00 CST     0.0     0.0
16  frontend2-0  mesh.source.frontend2.cartservice2.istio_reque...                       30              2022-03-20 11:00:00 CST     0.0     0.0
17  frontend2-0  mesh.source.frontend2.cartservice2.istio_reque...                       30              2022-03-20 11:00:00 CST     0.0     0.0
18  frontend2-0  mesh.source.frontend2.cartservice2.istio_reque...                       30              2022-03-20 11:00:00 CST     0.0     0.0
19  frontend2-0  mesh.source.frontend2.cartservice2.istio_reque...                       30              2022-03-20 11:00:00 CST     0.0     0.0,      node  total_anomaly_count_in_window  distinct_anomalous_series_count earliest_anomaly_timestamp
0  node-5                          51261                               64    2022-03-20 11:00:00 CST
1  node-6                          10792                               64    2022-03-20 11:00:00 CST)```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz

# reuse existing df_error and tz if available
try:
    df_error
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')
    df_error = pd.read_csv('error_logs.csv')
    df_error['ts'] = pd.to_datetime(df_error['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# ensure ts column exists and is tz-aware
if 'ts' not in df_error.columns:
    df_error['ts'] = pd.to_datetime(df_error['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Define frontends and narrow late window (inclusive)
frontends = {'frontend-0', 'frontend-1', 'frontend-2', 'frontend2-0'}
start = pd.Timestamp('2022-03-20 11:29:40', tz=tz)
end   = pd.Timestamp('2022-03-20 11:29:59', tz=tz)

# Filter and sort
slice_df = df_error[df_error['cmdb_id'].isin(frontends) & df_error['ts'].between(start, end)].copy()
slice_df = slice_df.sort_values('ts', ascending=True).head(50)

# Compact message helper (truncate to keep output readable)
def short_msg(s, maxlen=300):
    if pd.isna(s):
        return ''
    s = str(s).strip()
    return s if len(s) <= maxlen else s[:maxlen-3] + '...'

# Prepare display dataframe with formatted timestamp
if not slice_df.empty:
    display_df = slice_df.loc[:, ['ts','cmdb_id','message']].copy()
    display_df['timestamp'] = display_df['ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    display_df['message'] = display_df['message'].apply(short_msg)
    display_df = display_df[['timestamp','cmdb_id','message']].reset_index(drop=True)
else:
    display_df = pd.DataFrame(columns=['timestamp','cmdb_id','message'])

# One-line summary
total_rows = int(display_df.shape[0])
if total_rows > 0:
    earliest = display_df['timestamp'].iloc[0]
    latest = display_df['timestamp'].iloc[-1]
else:
    earliest = None
    latest = None

summary = f"earliest={earliest}, latest={latest}, total_rows_returned={total_rows}"

# Return compact results
display_df, summary

```
Out[7]:
```
summary = (
    "Filtered error logs (2022-03-20 11:29:40–11:29:59 UTC+8):\n"
    "- Rows returned: 20 (earliest 2022-03-20 11:29:40, latest 2022-03-20 11:29:59).\n"
    "- Affected pods in this slice: frontend-0, frontend-1, frontend-2 (frontend2-0 had no entries in this narrow window).\n"
    "- Message pattern: all entries are warnings with the text 'failed to retrieve...' (repeated across seconds in the 11:29:40–11:29:59 interval).\n"
    "- Interpretation: a short-lived but sustained series of retrieval warnings across multiple frontend pods occurred throughout the 20-second window (not a single isolated log line). Investigate the upstream retrieval target (timeouts/failures), network paths, or service(s) frontends depend on around 11:29:40–11:29:59."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(                  timestamp     cmdb_id                                            message
0   2022-03-20 11:29:40 CST  frontend-0  severity: warning, message: failed to retrieve...
1   2022-03-20 11:29:40 CST  frontend-1  severity: warning, message: failed to retrieve...
2   2022-03-20 11:29:41 CST  frontend-0  severity: warning, message: failed to retrieve...
3   2022-03-20 11:29:42 CST  frontend-1  severity: warning, message: failed to retrieve...
4   2022-03-20 11:29:42 CST  frontend-2  severity: warning, message: failed to retrieve...
5   2022-03-20 11:29:45 CST  frontend-2  severity: warning, message: failed to retrieve...
6   2022-03-20 11:29:45 CST  frontend-0  severity: warning, message: failed to retrieve...
7   2022-03-20 11:29:46 CST  frontend-2  severity: warning, message: failed to retrieve...
8   2022-03-20 11:29:48 CST  frontend-2  severity: warning, message: failed to retrieve...
9   2022-03-20 11:29:49 CST  frontend-0  severity: warning, message: failed to retrieve...
10  2022-03-20 11:29:50 CST  frontend-1  severity: warning, message: failed to retrieve...
11  2022-03-20 11:29:51 CST  frontend-0  severity: warning, message: failed to retrieve...
12  2022-03-20 11:29:52 CST  frontend-1  severity: warning, message: failed to retrieve...
13  2022-03-20 11:29:53 CST  frontend-2  severity: warning, message: failed to retrieve...
14  2022-03-20 11:29:54 CST  frontend-2  severity: warning, message: failed to retrieve...
15  2022-03-20 11:29:55 CST  frontend-1  severity: warning, message: failed to retrieve...
16  2022-03-20 11:29:55 CST  frontend-0  severity: warning, message: failed to retrieve...
17  2022-03-20 11:29:57 CST  frontend-2  severity: warning, message: failed to retrieve...
18  2022-03-20 11:29:57 CST  frontend-0  severity: warning, message: failed to retrieve...
19  2022-03-20 11:29:59 CST  frontend-2  severity: warning, message: failed to retrieve..., 'earliest=2022-03-20 11:29:40 CST, latest=2022-03-20 11:29:59 CST, total_rows_returned=20')```
```